# Algorithmic Systems Design: Smart Spell-Checker & Autocorrect Pipeline
**Authors:** Jakub Habib, Ulugbek Tojiboev

### System Objective
A real-time pipeline that verifies user text input and instantly suggests the top 3 statistically probable corrections for typos. The system chains three distinct algorithms to process data efficiently.

In [4]:
import re
import heapq

class DataPreparer:
    """Handles text sanitization and database loading."""
    @staticmethod
    def clean_text(text):
        """Removes punctuation and converts to lowercase."""
        if not text:
            return ""
        cleaned = re.sub(r'[^a-zA-Z]', '', text)
        return cleaned.lower()

    @staticmethod
    def load_database(raw_data, hash_table):
        """Cleans data and loads it into the provided hash table."""
        for word, freq in raw_data:
            cleaned = DataPreparer.clean_text(word)
            if cleaned:
                hash_table.Add(cleaned, freq)

class ChainingHashTable:
    """
    Stage 1: Hash Table with separate chaining and modulo hashing (O(1) access time).
    """
    def Init(self, p=100003):
        self.p = p
        self.table = [[] for _ in range(self.p)]
        
    def _hash_func(self, k_str):
        hash_val = 0
        for char in k_str:
            hash_val = (hash_val * 31 + ord(char)) % self.p
        return hash_val

    def Add(self, k, v):
        idx = self._hash_func(k)
        for i, (key, val) in enumerate(self.table[idx]):
            if key == k:
                self.table[idx][i] = (k, v)
                return
        self.table[idx].append((k, v))

    def Search(self, k):
        idx = self._hash_func(k)
        for key, val in self.table[idx]:
            if key == k:
                return True
        return False

    def GetValue(self, k):
        idx = self._hash_func(k)
        for key, val in self.table[idx]:
            if key == k:
                return val
        return 0
        
    def GetAllKeys(self):
        keys = []
        for bucket in self.table:
            for k, v in bucket:
                keys.append(k)
        return keys

In [5]:
class TopKMinHeap:
    """
    Stage 3: Min-Heap structure (fixed size k=3) for O(C log k) extraction.
    """
    def Init(self, k=3):
        self.k = k
        self.heap = []
        
    def Add(self, priority, value):
        """Adds candidate and removes the minimum if size exceeds k."""
        heapq.heappush(self.heap, (priority, value))
        if len(self.heap) > self.k:
            self.RemoveMin()
            
    def RemoveMin(self):
        """Removes the candidate with the lowest frequency."""
        return heapq.heappop(self.heap)
        
    def GetSortedResults(self):
        """Returns the top k words sorted by frequency (descending)."""
        result = sorted(self.heap, key=lambda x: x[0], reverse=True)
        return [val for priority, val in result]

In [ ]:
# --- MOCK TEST---
if __name__ == "__main__":
    mock_data = [
        ("the", 100000), 
        ("tea", 500), 
        ("ten", 3000), 
        ("tech", 15000), 
        ("ted", 150)
    ]
    
    # 1. Test Data Preparation & Hash Table
    hash_table = ChainingHashTable()
    hash_table.Init()
    DataPreparer.load_database(mock_data, hash_table)
    
    test_word = DataPreparer.clean_text("T.e,c!!h")
    print(f"Cleaned word: '{test_word}'")
    print(f"Is 'tech' in dictionary? {hash_table.Search(test_word)}")
    
    # 2. Test Min-Heap (simulating receiving candidates from Stage 2)
    heap = TopKMinHeap()
    heap.Init(k=3)
    
    print("\nProcessing candidates through Min-Heap...")
    for word in hash_table.GetAllKeys():
        freq = hash_table.GetValue(word)
        heap.Add(freq, word)
        
    print(f"Top 3 most frequent words: {heap.GetSortedResults()}")

Cleaned word: 'tech'
Is 'tech' in dictionary? True

Processing candidates through Min-Heap...
Top 3 most frequent words: ['the', 'tech', 'ten']
